# Step 2 &mdash; Dual-Stream Dataset (CLAHE + Tile + Global)

**Goal:** turn standardized ROIs into the two complementary views consumed by the dual-stream classifier:

- **Local stream** &mdash; fine-grained square tiles, capture micro-textures (scratches, grooving, flaking).
- **Global stream** &mdash; whole ROI resized to a wide strip, captures macro structure of the wear track.

Both streams share a CLAHE illumination-normalization step.

| Variant | Tile | Global view |
|---------|------|-------------|
| CNN  | 320 &times; 320 | 915 &times; 320  |
| Swin | 448 &times; 448 | 1280 &times; 448 |

## CLAHE illumination normalization

Raw inspection photos vary in lighting. Contrast-Limited Adaptive Histogram Equalization on the **L-channel** (LAB) normalizes brightness without destroying color information.

In [ ]:
import cv2

def apply_clahe(img, clip_limit=2.0, tile_grid=(8, 8)):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l_eq = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid).apply(l)
    return cv2.cvtColor(cv2.merge((l_eq, a, b)), cv2.COLOR_LAB2BGR)

## Local stream &mdash; grid tiles

Slide a fixed-size window across the image, optionally with overlap. The two outermost tile columns are dropped because they routinely contain ROI-edge artifacts that distract the classifier.

In [ ]:
def grid_tiles(img, tile_size=448, overlap=0.0, drop_edges=True):
    h, w = img.shape[:2]
    if h < tile_size or w < tile_size:
        return [cv2.resize(img, (tile_size, tile_size), interpolation=cv2.INTER_AREA)]

    stride = max(1, int(tile_size * (1.0 - overlap)))
    ys = list(range(0, h - tile_size + 1, stride)) or [0]
    xs = list(range(0, w - tile_size + 1, stride)) or [0]

    # Centering offset (distribute leftover margins symmetrically)
    off_y = (h - ((ys[-1] + tile_size) - ys[0])) // 2
    off_x = (w - ((xs[-1] + tile_size) - xs[0])) // 2

    tiles = []
    for y in ys:
        for c, x in enumerate(xs):
            if drop_edges and len(xs) > 2 and c in (0, len(xs) - 1):
                continue
            y0, x0 = y + off_y, x + off_x
            tiles.append(img[y0:y0 + tile_size, x0:x0 + tile_size])
    return tiles

## Global stream &mdash; single resized strip

In [ ]:
def global_view(img, width, height):
    return cv2.resize(img, (width, height), interpolation=cv2.INTER_AREA)

## Putting it together

For each standardized ROI we emit one global image and N tile images. A CSV manifest will be built in Step 3 once labels and splits are known.

Same probe identifier (image stem with `_STD` stripped) is shared across all of that probe's tiles &mdash; this is what allows probe-level multi-label splitting later.

In [ ]:
from pathlib import Path

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def get_probe_id(filename):
    return Path(filename).stem.replace("_STD", "").replace("_ROI", "")


def build_dual_stream(roi_dir, out_dir, tile_size, global_w, global_h):
    roi_dir, out_dir = Path(roi_dir), Path(out_dir)
    img_tiles = out_dir / "images_tiles"
    img_global = out_dir / "images_global"
    img_tiles.mkdir(parents=True, exist_ok=True)
    img_global.mkdir(parents=True, exist_ok=True)

    for p in sorted(roi_dir.rglob("*")):
        if p.suffix.lower() not in IMG_EXTS:
            continue
        img = cv2.imread(str(p))
        if img is None:
            continue
        img_eq = apply_clahe(img)
        pid = get_probe_id(p.name)

        cv2.imwrite(str(img_global / f"{pid}_global.jpg"),
                    global_view(img_eq, global_w, global_h))
        for i, tile in enumerate(grid_tiles(img_eq, tile_size)):
            cv2.imwrite(str(img_tiles / f"{pid}_tile{i}.jpg"), tile)

    print(f"Done. -> {out_dir}")


# Example (Swin variant)
# build_dual_stream("./outputs/swin/01_ROI", "./outputs/swin/02_DualStream", 448, 1280, 448)